# Experiment 2.20: Cosine Alignment Between Approximate Orthogonalization and the Newton Direction

## Scientific Motivation

In second-order optimization, the **Newton direction** $d_{\text{Newton}} = -H^{-1} g$ is the locally optimal descent step: it accounts for the full curvature landscape of the loss surface by inverting the Hessian $H$ and applying it to the gradient $g$. Any first-order optimizer can be evaluated by how well its update direction aligns with this gold-standard Newton step.

**Muon** (Momentum + Orthogonalization) computes its update by applying $k$ iterations of the **Newton-Schulz (NS) iterative polar decomposition** to each layer's gradient matrix $G$. The NS iteration converges to the orthogonal polar factor $U$ of $G = U \Sigma V^\top$, which is equivalent to replacing all singular values of $G$ with 1 (i.e., $U V^\top$). In practice, Muon uses $k = 5$ iterations -- far fewer than needed for exact convergence.

### The Core Question

This experiment asks: **Is $k = 5$ merely a computational shortcut, or is the inexactness itself beneficial?**

If approximate orthogonalization ($k \approx 5$) aligns *better* with the Newton direction than exact orthogonalization ($k \to \infty$), this would imply that the NS flow has an **optimal stopping point** -- a regime where it has removed gauge-redundant gradient components (rotational symmetries of the weight matrices) while *preserving* curvature-relevant information that exact orthogonalization destroys.

This connects to the **Renormalization Group (RG) gauge-fixing interpretation** of Muon: the NS iterations act as an RG flow that progressively integrates out irrelevant (gauge) degrees of freedom. Stopping at $k = 5$ corresponds to an optimal coarse-graining scale -- analogous to how in physics, the effective field theory at an intermediate energy scale can be more predictive than either the UV (raw gradient) or IR (exact polar factor) limits.

## Hypothesis

$$\cos\!\bigl(\text{ortho}_k(G),\; -H^{-1}g\bigr) \text{ peaks at } k \approx 3\text{--}5, \text{ then decreases.}$$

Specifically:
- $k = 5$ approximates the **natural gradient** (Newton direction) *better* than exact orthogonalization ($k = 20$, a proxy for $k \to \infty$).
- The cosine similarity should follow an **inverted-U** trajectory: rising from $k = 0$ (raw normalized gradient) as gauge noise is removed, peaking around $k = 3$--$5$, then declining as curvature information is over-projected away.

## Methodology

1. **Architecture:** 2-layer deep linear network with $4 \times 4$ weight matrices (32 total parameters). Deep linear networks have non-trivial loss landscapes (saddle points, gauge symmetries from layer rescaling) while remaining analytically tractable.
2. **Training trajectory:** SGD with momentum to a regime where loss $> 0$ and gradients are non-degenerate.
3. **Hessian computation:** Full $32 \times 32$ Hessian via central finite differences, yielding the exact Newton direction $d_{\text{Newton}} = -H^{\dagger} g$ (pseudoinverse to handle rank-deficient gauge directions).
4. **Sweep over $k$:** For each $k \in \{0, 1, 2, 3, 4, 5, 7, 10, 15, 20\}$, apply $k$ NS iterations to each layer's gradient, flatten to a 32-vector, and compute $\cos(d_{\text{Muon},k},\; d_{\text{Newton}})$.
5. **Baselines:** Compare against SGD (steepest descent $= -g$) and Adam update directions.
6. **Statistical robustness:** Repeat across 5 random seeds and 10 training checkpoints.

## Prior Context

- The **Newton-Schulz iteration** $X_{n+1} = \tfrac{3}{2} X_n - \tfrac{1}{2} X_n X_n^\top X_n$ converges cubically to the orthogonal polar factor for $\|X_0\| < 1$.
- The **Newton direction** is the theoretically optimal local step but is $O(n^3)$ to compute and numerically unstable for large networks.
- If approximate ortho ($k = 5$) beats exact ortho ($k = 20$) in Newton alignment, this constitutes evidence that **Muon's practical efficiency is not despite but because of its inexact orthogonalization** -- the "warm" regime preserves a beneficial curvature signal.

---

# Part I: Environment Setup and Experimental Configuration

Import required numerical libraries. This experiment is purely NumPy-based -- no deep learning framework is needed because we compute gradients analytically via backpropagation through the linear network and construct the full Hessian via finite differences. This keeps the computation transparent and verifiable.

In [ ]:
import numpy as np
import os
import sys

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


---

## Experimental Hyperparameters

The deep linear network uses $4 \times 4$ weight matrices across 2 layers, giving 32 total parameters. This is small enough to compute the full $32 \times 32$ Hessian exactly via finite differences (requiring $2 \times 32 = 64$ gradient evaluations per Hessian), while still exhibiting the key phenomena:

- **Gauge symmetry:** For any invertible $M$, the transformation $W_1 \to W_1 M,\; W_2 \to M^{-1} W_2$ leaves the product $W_2 W_1$ unchanged. This creates flat directions in the loss landscape that the Hessian's null space reflects.
- **Non-trivial curvature:** Despite linearity, the loss surface is a degree-4 polynomial in the weights (product of two matrices squared), producing genuine curvature variation across directions.

The NS iteration count $k$ is swept from 0 (normalized gradient, no orthogonalization) through 20 (near-converged orthogonal polar factor). The critical test range is $k = 3$--$5$, where we hypothesize peak Newton alignment.

In [ ]:
DIM = 4                    # 4x4 matrices => 16 params per layer
NUM_LAYERS = 2             # 2-layer deep linear net => 32 total params
N_PARAMS = NUM_LAYERS * DIM * DIM  # 32
HESSIAN_EPS = 1e-5         # Finite difference step for Hessian
NS_K_VALUES = [0, 1, 2, 3, 4, 5, 7, 10, 15, 20]

print("\n--- Experimental Configuration ---")
print(f"  DIM = {DIM}")
print(f"  NUM_LAYERS = {NUM_LAYERS}")
print(f"  N_PARAMS = {N_PARAMS} (total learnable parameters)")
print(f"  HESSIAN_EPS = {HESSIAN_EPS} (central finite-difference step)")
print(f"  NS_K_VALUES = {NS_K_VALUES}")
print(f"\n  Hessian size: {N_PARAMS} x {N_PARAMS} = {N_PARAMS**2} entries")
print(f"  Finite-difference gradient evaluations per Hessian: {2 * N_PARAMS}")
print(f"  Gauge degrees of freedom (layer rescaling): dim^2 = {DIM**2} directions in null(H)")

### Training and Measurement Schedule

We train with SGD+momentum (a simple, well-understood optimizer) and take measurements at logarithmically-spaced training steps. Early steps probe the high-loss regime (strong gradients, potentially ill-conditioned Hessian), while later steps probe the low-loss regime (weaker gradients, better-conditioned landscape near a minimum). This lets us check whether the Newton-alignment phenomenon is robust across the training trajectory or specific to a particular loss regime.

In [ ]:
# Training steps at which to measure alignment
MEASUREMENT_STEPS = [10, 20, 50, 100, 200, 300, 500, 750, 1000, 1500]
TRAIN_LR = 0.005           # Small LR for SGD pre-training (keep gradients nonzero)
MOMENTUM = 0.9
NUM_SEEDS = 5              # Average over seeds for robustness

print(f"  MEASUREMENT_STEPS = {MEASUREMENT_STEPS}")
print(f"  TRAIN_LR = {TRAIN_LR} (deliberately small to keep loss > 0 at measurement points)")
print(f"  MOMENTUM = {MOMENTUM}")
print(f"  NUM_SEEDS = {NUM_SEEDS}")
print(f"  Total measurement points (max): {NUM_SEEDS} seeds x {len(MEASUREMENT_STEPS)} steps = {NUM_SEEDS * len(MEASUREMENT_STEPS)}")

### Baseline Optimizer: Adam

Adam is included as a second baseline alongside SGD. Adam maintains exponential moving averages of the first and second moments of the gradient, producing an update direction $m_t / (\sqrt{v_t} + \epsilon)$ that adapts per-parameter learning rates. This is a form of diagonal preconditioning -- it approximates the Newton direction under the assumption that the Hessian is diagonal. Comparing Muon's Newton alignment against Adam's tells us whether orthogonalization captures *more* curvature structure than diagonal rescaling alone.

In [ ]:
# Adam hyperparameters (for baseline comparison)
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPS = 1e-8

print(f"  Adam baseline: beta1={ADAM_BETA1}, beta2={ADAM_BETA2}, eps={ADAM_EPS}")
print(f"  Note: Adam state is tracked but NOT used for weight updates --")
print(f"        we only extract its update direction for cosine comparison.")

---

# Part II: Network Architecture and Gradient Computation

## Weight Initialization

Weights are initialized near the identity matrix ($W = I + 0.1 \cdot \mathcal{N}(0,1)$). This ensures:
- The initial product $W_2 W_1 \approx I$, so the network starts close to the identity map.
- The loss landscape is locally well-conditioned at initialization (no degenerate singular values).
- The perturbation breaks exact symmetry, ensuring non-trivial gradient directions from the start.

In [ ]:
def init_weights(dim, num_layers, seed):
    """Initialize layers near identity."""
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        W = np.eye(dim) + rng.randn(dim, dim) * 0.1
        weights.append(W.copy())
    return weights

## Forward Pass

The deep linear network computes $Y = W_2 \cdot W_1 \cdot X$. Although the composition of linear maps is itself linear (the network is equivalent to a single matrix $W_{\text{eff}} = W_2 W_1$), the loss surface as a function of the *individual* weight matrices is **non-convex**. The parameterization into separate layers introduces:

- **Saddle points** at rank-deficient solutions where $\text{rank}(W_{\text{eff}}) < \min(d_{\text{in}}, d_{\text{out}})$.
- **Gauge symmetries** (flat directions) from the layer-rescaling invariance $W_1 \to W_1 M, W_2 \to M^{-1} W_2$.
- **Curved valleys** where the Hessian has a rich eigenspectrum despite the underlying linearity.

This makes deep linear networks an ideal testbed: they are analytically tractable yet exhibit the geometric structure (gauge symmetry, curvature anisotropy) that motivates Muon's design.

In [ ]:
def forward_linear(weights, X):
    """Forward pass through deep linear net."""
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

## Loss Function

The loss is the standard Mean Squared Error (MSE):

$$\mathcal{L}(W_1, W_2) = \frac{1}{2N} \sum_{i=1}^{N} \| W_2 W_1 x_i - y_i \|^2$$

where $y_i = W_2^* W_1^* x_i$ are targets generated from random target matrices $W_1^*, W_2^*$. The factor of $\frac{1}{2}$ simplifies gradient expressions. Since the network output is $W_2 W_1 X$, the loss is a quartic polynomial in the weight entries -- the simplest setting that produces non-trivial Hessian structure.

In [ ]:
def compute_loss(weights, X, Y_target):
    """MSE loss."""
    Y_pred = forward_linear(weights, X)
    diff = Y_pred - Y_target
    return 0.5 * np.mean(diff ** 2)

## Analytical Gradient via Backpropagation

Gradients are computed analytically (not via finite differences) using the chain rule through the linear layers. For layer $\ell$:

$$\nabla_{W_\ell} \mathcal{L} = \delta_\ell \cdot a_{\ell-1}^\top$$

where $a_{\ell-1}$ is the input activation to layer $\ell$ and $\delta_\ell$ is the error signal backpropagated from the output. Each gradient $\nabla_{W_\ell} \mathcal{L}$ is a $4 \times 4$ matrix -- this matrix structure is what Muon's orthogonalization operates on, treating each layer's gradient as a linear map rather than a flat vector.

In [ ]:
def compute_gradients(weights, X, Y_target):
    """Backprop through deep linear net. Returns list of gradient matrices."""
    num_layers = len(weights)
    batch_size = X.shape[1]

    # Forward pass -- store activations
    activations = [X.copy()]
    for W in weights:
        activations.append(W @ activations[-1])

    # Output error
    Y_pred = activations[-1]
    delta = (Y_pred - Y_target) / batch_size

    # Backward pass
    grads = [None] * num_layers
    for l in range(num_layers - 1, -1, -1):
        grads[l] = delta @ activations[l].T
        if l > 0:
            delta = weights[l].T @ delta

    return grads

---

## Parameter Space Flattening Utilities

To compare optimizer directions with the Newton direction, we need to work in the flat $\mathbb{R}^{32}$ parameter space. These utilities convert between the "structured" representation (list of $4 \times 4$ matrices, one per layer) and the "flat" representation (single 32-vector). The flattening order is layer 0 first, then layer 1, with each matrix flattened in row-major order.

This distinction matters: Muon operates on the **matrix** representation (applying NS per-layer), while Newton operates on the **flat vector** representation (inverting the global Hessian). The cosine similarity between these two fundamentally different geometric objects is the central measurement of this experiment.

In [ ]:
def weights_to_vector(weights):
    """Flatten all weight matrices into a single vector."""
    return np.concatenate([W.flatten() for W in weights])

In [ ]:
def vector_to_weights(vec, dim, num_layers):
    """Unflatten a vector back into list of weight matrices."""
    weights = []
    idx = 0
    for _ in range(num_layers):
        size = dim * dim
        W = vec[idx:idx + size].reshape(dim, dim)
        weights.append(W)
        idx += size
    return weights

In [ ]:
def grads_to_vector(grads):
    """Flatten gradient matrices into a single vector."""
    return np.concatenate([g.flatten() for g in grads])

---

# Part III: The Newton-Schulz Orthogonalization (Core of Muon)

## Newton-Schulz Iteration

The Newton-Schulz (NS) iteration is the heart of Muon. Starting from $X_0 = G / \|G\|_F$ (Frobenius-normalized gradient), it iterates:

$$X_{n+1} = \tfrac{3}{2} X_n - \tfrac{1}{2} (X_n X_n^\top) X_n$$

This converges cubically to the **orthogonal polar factor** $U$ of $G = U \Sigma V^\top$, which satisfies $U U^\top = I$. The polar factor replaces all singular values with 1 while preserving the singular vector structure.

### What each iteration does geometrically:

| Iteration $k$ | Effect |
|:-:|:--|
| $k = 0$ | Frobenius-normalized gradient $G / \|G\|_F$. All singular value ratios preserved. |
| $k = 1$ | Singular values pushed toward 1, but large ratios remain. Mild equalization. |
| $k = 2$--$3$ | Substantial equalization. Small singular values amplified, large ones suppressed. |
| $k = 4$--$5$ | Near-convergence for well-conditioned $G$. Residual deviation $O(10^{-3})$ from orthogonal. |
| $k = 10$+ | Effectively exact orthogonal polar factor for any reasonable condition number. |

The hypothesis is that the intermediate regime ($k \approx 5$) preserves gradient components that carry curvature information (about the Hessian eigenstructure), while the fully-converged regime ($k \to \infty$) projects these away by forcing all singular values to exactly 1.

In [ ]:
def newton_schulz_orthogonalize(G, num_iters):
    """Apply Newton-Schulz iteration to approximate orthogonal polar factor.

    k=0 means just the normalized gradient (no NS iterations).
    """
    norm = np.linalg.norm(G, ord='fro')
    if norm < 1e-12:
        return G
    X = G / norm

    for _ in range(num_iters):
        A = X @ X.T
        X = 1.5 * X - 0.5 * A @ X

    return X

---

# Part IV: Second-Order Analysis -- Hessian and Newton Direction

## Full Hessian via Central Finite Differences

The Hessian $H_{ij} = \partial^2 \mathcal{L} / \partial \theta_i \partial \theta_j$ is computed column-by-column using central finite differences on the gradient:

$$H_{:,i} \approx \frac{\nabla_\theta \mathcal{L}(\theta + \epsilon e_i) - \nabla_\theta \mathcal{L}(\theta - \epsilon e_i)}{2\epsilon}$$

This requires $2 \times 32 = 64$ gradient evaluations for our 32-parameter network. The result is symmetrized as $H \leftarrow \frac{1}{2}(H + H^\top)$ to eliminate numerical asymmetry from floating-point errors.

**Key properties of this Hessian for deep linear networks:**
- It contains **gauge null directions** corresponding to the layer-rescaling symmetry $W_1 \to W_1 M, W_2 \to M^{-1} W_2$. These directions have zero eigenvalue and the gradient has zero component along them (by Noether's theorem).
- The non-zero eigenvalues encode the **curvature of the loss along physically meaningful directions** -- precisely the information that an ideal optimizer should exploit.
- We use `np.linalg.pinv` (pseudoinverse) to compute the Newton direction, which automatically projects out the null space.

In [ ]:
def compute_gradient_vector(weights, X, Y_target):
    """Return gradient as a flat vector."""
    grads = compute_gradients(weights, X, Y_target)
    return grads_to_vector(grads)

## Hessian Computation Implementation

The function below constructs the full $32 \times 32$ Hessian matrix. Note the use of central differences (rather than forward differences) for $O(\epsilon^2)$ accuracy instead of $O(\epsilon)$. The step size $\epsilon = 10^{-5}$ balances truncation error (decreases with smaller $\epsilon$) against floating-point cancellation error (increases with smaller $\epsilon$).

In [ ]:
def compute_full_hessian(weights, X, Y_target, eps=HESSIAN_EPS):
    """Compute full Hessian via central finite differences on the gradient."""
    theta = weights_to_vector(weights)
    n_params = len(theta)

    H = np.zeros((n_params, n_params))

    for i in range(n_params):
        theta_plus = theta.copy()
        theta_minus = theta.copy()
        theta_plus[i] += eps
        theta_minus[i] -= eps

        w_plus = vector_to_weights(theta_plus, DIM, NUM_LAYERS)
        w_minus = vector_to_weights(theta_minus, DIM, NUM_LAYERS)

        grad_plus = compute_gradient_vector(w_plus, X, Y_target)
        grad_minus = compute_gradient_vector(w_minus, X, Y_target)

        H[:, i] = (grad_plus - grad_minus) / (2 * eps)

    # Symmetrize
    H = 0.5 * (H + H.T)
    return H

---

## Cosine Similarity as Directional Alignment Metric

The cosine similarity $\cos(\theta) = \frac{a \cdot b}{\|a\| \|b\|}$ measures pure directional alignment, ignoring magnitude. This is the correct metric because:

- We care about **which direction** the optimizer moves, not **how far** (that is controlled by the learning rate).
- A cosine of $+1$ means perfect alignment with the Newton direction; $0$ means orthogonal (no useful curvature information captured); $-1$ means anti-aligned.
- For a 32-dimensional space, two random unit vectors have expected cosine $\sim 0$ with standard deviation $\sim 1/\sqrt{32} \approx 0.18$. Any consistent positive cosine above $\sim 0.3$ is statistically meaningful.

In [ ]:
def cosine_sim(a, b):
    """Cosine similarity between two vectors. Returns 0 if either is zero."""
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < 1e-15 or nb < 1e-15:
        return 0.0
    return np.dot(a, b) / (na * nb)

---

# Part V: Optimizer Direction Computation

## Muon Direction for a Given $k$

This function implements the core of what Muon does: apply $k$ Newton-Schulz iterations to each layer's gradient matrix independently, then flatten the results into a single parameter-space vector. The per-layer independence is critical -- Muon treats each weight matrix as a separate linear map and orthogonalizes its gradient in the matrix space $\mathbb{R}^{m \times n}$, not in the flat parameter space $\mathbb{R}^{mn}$.

At $k = 0$, the output is simply the Frobenius-normalized gradient (equivalent to normalized SGD applied per-layer). As $k$ increases, the singular values of each gradient matrix are progressively equalized toward 1, changing the *direction* in parameter space.

In [ ]:
def compute_muon_direction(grads, k):
    """Apply k NS iterations to each gradient matrix, then flatten.

    k=0 returns normalized gradient (no NS, just Frobenius normalization per layer).
    """
    direction_parts = []
    for G in grads:
        G_orth = newton_schulz_orthogonalize(G, num_iters=k)
        direction_parts.append(G_orth.flatten())
    return np.concatenate(direction_parts)

## Adam Direction (Stateless Snapshot)

Adam's update direction is computed from the accumulated first moment $m_t$ and second moment $v_t$, with bias correction. This is a **diagonal preconditioner**: it rescales each parameter's gradient by $1/\sqrt{v_t + \epsilon}$, which approximates the Newton direction only if the Hessian is diagonal. For deep linear networks with gauge symmetry, the Hessian is far from diagonal, so Adam's alignment with Newton should be limited compared to Muon's matrix-aware orthogonalization.

Note: The Adam state is maintained in parallel with SGD training but is **not used to update weights**. We only extract the direction Adam *would* move in, for comparison purposes.

In [ ]:
def compute_adam_direction(m_state, v_state, t):
    """Compute Adam update direction from current moment estimates."""
    m_hat = m_state / (1 - ADAM_BETA1 ** t)
    v_hat = v_state / (1 - ADAM_BETA2 ** t)
    direction = m_hat / (np.sqrt(v_hat) + ADAM_EPS)
    return direction

---

# Part VI: Core Measurement -- Alignment at a Training Checkpoint

## Single-Point Alignment Measurement

At each training checkpoint, this function performs the full measurement pipeline:

1. **Compute the gradient** $g \in \mathbb{R}^{32}$ -- the raw first-order signal.
2. **Construct the full Hessian** $H \in \mathbb{R}^{32 \times 32}$ via finite differences.
3. **Compute the Newton direction** $d_{\text{Newton}} = -H^{\dagger} g$ using the pseudoinverse (to handle gauge null directions gracefully).
4. **Sweep $k$:** For each NS iteration count, compute the Muon direction and its cosine with Newton.
5. **Baselines:** Compute SGD ($-g$) and Adam ($-m_t / \sqrt{v_t + \epsilon}$) cosines with Newton.
6. **Diagnostics:** Record Hessian condition number and effective rank for interpretive context.

The Hessian eigendecomposition provides crucial context: a high condition number means the curvature landscape is highly anisotropic, which is precisely the regime where second-order methods (and Muon) should shine relative to SGD.

In [ ]:
def measure_alignment(weights, X, Y_target, adam_m, adam_v, adam_t):
    """At current weights, compute Newton direction and cosine with ortho_k.

    Returns dict mapping k -> cosine, plus SGD and Adam cosines.
    """
    # 1. Compute gradient (flat vector)
    grads = compute_gradients(weights, X, Y_target)
    g = grads_to_vector(grads)
    g_norm = np.linalg.norm(g)

    if g_norm < 1e-15:
        # Gradient vanished -- skip
        print("      [measure_alignment] Gradient norm < 1e-15, skipping (converged or saddle)")
        return None

    # 2. Compute full Hessian
    H = compute_full_hessian(weights, X, Y_target)

    # 3. Newton direction via pseudoinverse (handles singular/gauge directions)
    H_pinv = np.linalg.pinv(H, rcond=1e-10)
    d_newton = -H_pinv @ g
    d_newton_norm = np.linalg.norm(d_newton)

    if d_newton_norm < 1e-15:
        print("      [measure_alignment] Newton direction norm < 1e-15, skipping")
        return None

    # Print intermediate second-order diagnostics
    print(f"      [Hessian] ||g|| = {g_norm:.6e}, ||d_Newton|| = {d_newton_norm:.6e}")
    print(f"      [Hessian] H symmetry check: ||H - H^T||_F = {np.linalg.norm(H - H.T):.2e}")

    # 4. Cosine with ortho_k(G) for each k
    cos_by_k = {}
    for k in NS_K_VALUES:
        d_muon_k = compute_muon_direction(grads, k)
        # Muon direction is negative (descent)
        cos_by_k[k] = cosine_sim(-d_muon_k, d_newton)

    # Print the cosine sweep for this measurement point
    sweep_str = ", ".join([f"k={k}:{cos_by_k[k]:+.3f}" for k in [0, 3, 5, 10, 20]])
    print(f"      [Cosine sweep] {sweep_str}")

    # 5. SGD direction = -g (steepest descent)
    cos_sgd = cosine_sim(-g, d_newton)

    # 6. Adam direction
    if adam_t > 0:
        adam_dir = compute_adam_direction(adam_m, adam_v, adam_t)
        cos_adam = cosine_sim(-adam_dir, d_newton)
    else:
        cos_adam = cos_sgd  # At t=0, Adam = SGD

    # 7. Also compute Hessian condition info for context
    eigenvalues = np.linalg.eigvalsh(H)
    lambda_max = np.max(np.abs(eigenvalues))
    n_significant = np.sum(np.abs(eigenvalues) > 0.01 * lambda_max) if lambda_max > 1e-15 else 0

    # Print Hessian spectral diagnostics
    n_positive = np.sum(eigenvalues > 1e-12)
    n_negative = np.sum(eigenvalues < -1e-12)
    n_zero = len(eigenvalues) - n_positive - n_negative
    print(f"      [Spectrum] lambda_max={lambda_max:.4e}, "
          f"eigenvalue signs: {n_positive}+, {n_negative}-, {n_zero} null "
          f"(expect ~{DIM**2} gauge null directions)")

    return {
        'cos_by_k': cos_by_k,
        'cos_sgd': cos_sgd,
        'cos_adam': cos_adam,
        'loss': compute_loss(weights, X, Y_target),
        'grad_norm': g_norm,
        'newton_norm': d_newton_norm,
        'hessian_cond': lambda_max / max(np.min(np.abs(eigenvalues[np.abs(eigenvalues) > 1e-12])), 1e-15) if np.any(np.abs(eigenvalues) > 1e-12) else np.inf,
        'hessian_rank': int(n_significant),
    }

---

# Part VII: Training Loop with Embedded Measurements

## Single-Seed Training Run

For each random seed, the experiment:
1. Generates random target matrices $W_1^*, W_2^*$ and training data $X$.
2. Initializes the network weights near identity.
3. Trains with SGD+momentum (the simplest possible optimizer) while maintaining Adam state in parallel.
4. At each measurement step, pauses training to compute the full alignment profile: Newton direction, Muon direction for all $k$ values, and baseline directions.

The SGD training ensures we traverse a representative region of the loss landscape. The small learning rate ($0.005$) keeps the network from converging too quickly, ensuring we have measurements at multiple loss scales.

In [ ]:
def run_single_seed(seed):
    """Train with SGD, measure alignment at specified steps."""
    rng = np.random.RandomState(seed)

    # Generate random target
    W_target = [rng.randn(DIM, DIM) * 0.5 for _ in range(NUM_LAYERS)]
    X = rng.randn(DIM, 32) * 0.5  # 32 data points

    Y_target = X.copy()
    for W in W_target:
        Y_target = W @ Y_target

    # Initialize weights
    weights = init_weights(DIM, NUM_LAYERS, seed + 1000)

    # Print initialization diagnostics
    W_eff_init = weights[1] @ weights[0]
    W_eff_target = W_target[1] @ W_target[0]
    sv_init = np.linalg.svd(W_eff_init, compute_uv=False)
    sv_target = np.linalg.svd(W_eff_target, compute_uv=False)
    print(f"    [Init] W_eff singular values: {np.array2string(sv_init, precision=3)}")
    print(f"    [Init] W_target singular values: {np.array2string(sv_target, precision=3)}")

    # SGD momentum buffer
    velocities = [np.zeros_like(W) for W in weights]

    # Adam state (tracked in flat param space for direction computation)
    adam_m = np.zeros(N_PARAMS)
    adam_v = np.zeros(N_PARAMS)
    adam_t = 0

    # Check initial loss
    initial_loss = compute_loss(weights, X, Y_target)
    print(f"    [Init] Initial loss: {initial_loss:.6e}")

    measurements = []
    max_step = max(MEASUREMENT_STEPS) + 1

    for step in range(1, max_step + 1):
        # Compute gradients
        grads = compute_gradients(weights, X, Y_target)
        g_flat = grads_to_vector(grads)

        # Update Adam state (for direction comparison only)
        adam_t += 1
        adam_m = ADAM_BETA1 * adam_m + (1 - ADAM_BETA1) * g_flat
        adam_v = ADAM_BETA2 * adam_v + (1 - ADAM_BETA2) * (g_flat ** 2)

        # SGD + momentum update
        for i in range(len(weights)):
            velocities[i] = MOMENTUM * velocities[i] + grads[i]
            weights[i] = weights[i] - TRAIN_LR * velocities[i]

        # Measure at specified steps
        if step in MEASUREMENT_STEPS:
            loss_now = compute_loss(weights, X, Y_target)
            if loss_now < 1e-12:
                # Loss too small, gradients effectively zero
                print(f"    Step {step}: loss={loss_now:.2e} (too small, skipping)")
                continue

            result = measure_alignment(weights, X, Y_target, adam_m, adam_v, adam_t)
            if result is not None:
                result['step'] = step
                measurements.append(result)
                print(f"    Step {step:5d}: loss={result['loss']:.4e}, "
                      f"|g|={result['grad_norm']:.4e}, "
                      f"H_rank={result['hessian_rank']}, "
                      f"cos(k=5,Newt)={result['cos_by_k'][5]:+.4f}")
            else:
                print(f"    Step {step:5d}: gradient or Newton direction vanished, skipping")

    # Print seed summary
    if measurements:
        avg_cos5 = np.mean([m['cos_by_k'][5] for m in measurements])
        avg_cos20 = np.mean([m['cos_by_k'][20] for m in measurements])
        avg_sgd = np.mean([m['cos_sgd'] for m in measurements])
        print(f"    [Seed summary] {len(measurements)} valid measurements")
        print(f"    [Seed summary] avg cos(k=5, Newton)={avg_cos5:+.4f}, "
              f"cos(k=20, Newton)={avg_cos20:+.4f}, "
              f"cos(SGD, Newton)={avg_sgd:+.4f}")
        print(f"    [Seed summary] k=5 vs k=20 gap: {avg_cos5 - avg_cos20:+.4f} "
              f"({'approximate ortho WINS' if avg_cos5 > avg_cos20 else 'exact ortho wins'})")

    return measurements

---

# Part VIII: Statistical Aggregation and Reporting

## Aggregation Across Seeds and Training Steps

Results are aggregated in two ways:
1. **Global average:** Mean and standard deviation of $\cos(\text{ortho}_k, \text{Newton})$ across all measurement points (seeds $\times$ steps). This gives the overall picture.
2. **Per-step breakdown:** Mean cosine at each training step, averaged across seeds. This reveals whether the alignment pattern changes as the network trains (e.g., whether the optimal $k$ shifts as the loss landscape evolves).

In [ ]:
def aggregate_results(all_measurements):
    """Aggregate cosine measurements across seeds and steps.

    Returns:
      - avg_cos_by_k: dict k -> mean cosine across all measurement points
      - std_cos_by_k: dict k -> std cosine
      - avg_cos_sgd: mean cosine of SGD direction
      - avg_cos_adam: mean cosine of Adam direction
      - per_step_data: dict step -> {k -> [cosines across seeds]}
    """
    # Flatten all measurements
    all_cos_by_k = {k: [] for k in NS_K_VALUES}
    all_cos_sgd = []
    all_cos_adam = []

    per_step_data = {}

    for seed_measurements in all_measurements:
        for m in seed_measurements:
            step = m['step']
            if step not in per_step_data:
                per_step_data[step] = {k: [] for k in NS_K_VALUES}
                per_step_data[step]['sgd'] = []
                per_step_data[step]['adam'] = []

            for k in NS_K_VALUES:
                all_cos_by_k[k].append(m['cos_by_k'][k])
                per_step_data[step][k].append(m['cos_by_k'][k])

            all_cos_sgd.append(m['cos_sgd'])
            all_cos_adam.append(m['cos_adam'])
            per_step_data[step]['sgd'].append(m['cos_sgd'])
            per_step_data[step]['adam'].append(m['cos_adam'])

    avg_cos_by_k = {k: np.mean(all_cos_by_k[k]) for k in NS_K_VALUES}
    std_cos_by_k = {k: np.std(all_cos_by_k[k]) for k in NS_K_VALUES}

    return {
        'avg_cos_by_k': avg_cos_by_k,
        'std_cos_by_k': std_cos_by_k,
        'avg_cos_sgd': np.mean(all_cos_sgd),
        'std_cos_sgd': np.std(all_cos_sgd),
        'avg_cos_adam': np.mean(all_cos_adam),
        'std_cos_adam': np.std(all_cos_adam),
        'per_step_data': per_step_data,
        'n_measurements': len(all_cos_sgd),
    }

In [ ]:
def print_separator(char='=', width=100):
    print(char * width)

---

# Part IX: Main Execution -- Hypothesis Testing and Interpretation

## Main Function

The `main()` function orchestrates the full experiment:

1. **Training phase:** Runs `NUM_SEEDS` independent training trajectories, collecting alignment measurements at each checkpoint.
2. **Table 1 (Global):** Displays the overall average $\cos(\text{ortho}_k, \text{Newton})$ for each $k$, revealing the shape of the alignment curve (inverted-U vs. monotonic).
3. **Table 2 (Per-step):** Shows how the alignment evolves over training, checking for consistency.
4. **Hypothesis tests:** Four quantitative tests of the central predictions:
   - **Test 1:** Does the peak cosine occur at $k = 3$--$5$?
   - **Test 2:** Is $k = 5$ better than $k = 20$ (approximate vs. exact ortho)?
   - **Test 3:** Does cosine decline after the peak (over-orthogonalization)?
   - **Test 4:** Does Muon beat SGD and Adam baselines?
5. **Interpretation:** Synthesizes results into a verdict on the RG gauge-fixing hypothesis.

In [ ]:
def main():
    print()
    print_separator('#')
    print("  EXPERIMENT 2.20: cos(ortho_k(G), Newton direction) vs k")
    print("  Does Muon's approximate ortho (k=5) align BETTER with Newton than exact ortho?")
    print_separator('#')
    print()
    print(f"  Network: {NUM_LAYERS}-layer deep linear, {DIM}x{DIM} = {N_PARAMS} params")
    print(f"  Training: SGD+momentum (lr={TRAIN_LR}, mom={MOMENTUM})")
    print(f"  Measurement steps: {MEASUREMENT_STEPS}")
    print(f"  NS k values: {NS_K_VALUES}")
    print(f"  Seeds: {NUM_SEEDS}")
    print(f"  Hessian: central finite differences (eps={HESSIAN_EPS})")
    print()

    all_measurements = []

    for seed_idx in range(NUM_SEEDS):
        seed = 42 + seed_idx * 137
        print_separator('-')
        print(f"  Seed {seed_idx+1}/{NUM_SEEDS} (seed={seed})")
        print_separator('-')
        measurements = run_single_seed(seed)
        all_measurements.append(measurements)
        print()

    # Aggregate
    agg = aggregate_results(all_measurements)

    # ==========================================================================
    # TABLE 1: Overall average cos(ortho_k, Newton) vs k
    # ==========================================================================
    print_separator('=')
    print("  TABLE 1: cos(ortho_k(G), Newton direction) averaged over ALL measurement points")
    print(f"  ({agg['n_measurements']} measurements = {NUM_SEEDS} seeds x up to {len(MEASUREMENT_STEPS)} steps)")
    print_separator('=')
    print()
    print(f"  {'k':<6s} | {'cos(ortho_k, Newton)':<24s} | {'std':<10s} | {'bar':<40s}")
    print(f"  {'-'*6}-+-{'-'*24}-+-{'-'*10}-+-{'-'*40}")

    best_k = max(NS_K_VALUES, key=lambda k: agg['avg_cos_by_k'][k])
    max_cos = max(agg['avg_cos_by_k'].values())

    for k in NS_K_VALUES:
        cos_val = agg['avg_cos_by_k'][k]
        std_val = agg['std_cos_by_k'][k]
        # Visual bar (scale 0 to max_cos)
        bar_len = int(35 * max(0, cos_val) / max(max_cos, 0.01))
        bar = '#' * bar_len
        marker = " <<< PEAK" if k == best_k else ""
        print(f"  k={k:<3d} | {cos_val:>+.6f}                | {std_val:.6f} | {bar}{marker}")

    print()
    print(f"  SGD direction (steepest descent):")
    print(f"         | {agg['avg_cos_sgd']:>+.6f}                | {agg['std_cos_sgd']:.6f} |")
    print(f"  Adam direction:")
    print(f"         | {agg['avg_cos_adam']:>+.6f}                | {agg['std_cos_adam']:.6f} |")
    print()

    # ==========================================================================
    # TABLE 2: Per-step breakdown
    # ==========================================================================
    print_separator('=')
    print("  TABLE 2: cos(ortho_k, Newton) per training step (averaged over seeds)")
    print_separator('=')
    print()

    # Header
    header = f"  {'step':<6s}"
    for k in NS_K_VALUES:
        header += f" | k={k:<3d} "
    header += f" | {'SGD':>7s} | {'Adam':>7s}"
    print(header)
    print(f"  {'-'*6}" + "".join([f"-+-{'-'*7}" for _ in NS_K_VALUES]) + f"-+-{'-'*7}-+-{'-'*7}")

    sorted_steps = sorted(agg['per_step_data'].keys())
    for step in sorted_steps:
        sd = agg['per_step_data'][step]
        row = f"  {step:<6d}"
        for k in NS_K_VALUES:
            if len(sd[k]) > 0:
                row += f" | {np.mean(sd[k]):+.4f}"
            else:
                row += f" |    N/A"
        if len(sd['sgd']) > 0:
            row += f" | {np.mean(sd['sgd']):+.4f}"
            row += f" | {np.mean(sd['adam']):+.4f}"
        else:
            row += f" |    N/A |    N/A"
        print(row)

    print()

    # ==========================================================================
    # HYPOTHESIS TEST
    # ==========================================================================
    print_separator('*')
    print("  HYPOTHESIS TEST")
    print_separator('*')
    print()

    # Test 1: Does cosine peak at k=3-5?
    cos_vals = [(k, agg['avg_cos_by_k'][k]) for k in NS_K_VALUES]
    peak_k, peak_cos = max(cos_vals, key=lambda x: x[1])

    print(f"  1. Peak cosine occurs at k={peak_k} (cos={peak_cos:+.6f})")
    if 3 <= peak_k <= 5:
        print(f"     >>> CONFIRMED: Peak is in predicted range k=3-5")
    elif 1 <= peak_k <= 7:
        print(f"     >>> PARTIAL: Peak at k={peak_k} is near but not exactly k=3-5")
    else:
        print(f"     >>> NOT CONFIRMED: Peak at k={peak_k} is outside predicted range k=3-5")
    print()

    # Test 2: Is k=5 better than k=20?
    cos_5 = agg['avg_cos_by_k'][5]
    cos_20 = agg['avg_cos_by_k'][20]
    print(f"  2. cos(ortho_5, Newton) = {cos_5:+.6f}")
    print(f"     cos(ortho_20, Newton) = {cos_20:+.6f}")
    print(f"     Difference: {cos_5 - cos_20:+.6f}")
    if cos_5 > cos_20:
        print(f"     >>> CONFIRMED: k=5 is CLOSER to Newton than k=20 (exact ortho)")
        print(f"     Interpretation: Approximate ortho preserves curvature info that exact ortho destroys")
    else:
        print(f"     >>> NOT CONFIRMED: k=20 is closer to Newton than k=5")
    print()

    # Test 3: Does cosine decline after peak?
    if peak_k < max(NS_K_VALUES):
        post_peak = [(k, agg['avg_cos_by_k'][k]) for k in NS_K_VALUES if k > peak_k]
        if len(post_peak) > 0:
            last_cos = post_peak[-1][1]
            print(f"  3. After peak (k={peak_k}): cos declines from {peak_cos:+.6f} to {last_cos:+.6f}")
            if last_cos < peak_cos:
                print(f"     >>> CONFIRMED: Cosine decreases after peak (over-orthogonalization hurts)")
            else:
                print(f"     >>> NOT CONFIRMED: Cosine does not decrease after peak")
    else:
        print(f"  3. Peak is at max k tested -- cannot assess decline")
    print()

    # Test 4: Muon vs baselines
    cos_sgd = agg['avg_cos_sgd']
    cos_adam = agg['avg_cos_adam']
    print(f"  4. Baseline comparisons:")
    print(f"     cos(SGD,  Newton) = {cos_sgd:+.6f}")
    print(f"     cos(Adam, Newton) = {cos_adam:+.6f}")
    print(f"     cos(k=5,  Newton) = {cos_5:+.6f}")
    print(f"     cos(k={peak_k} [best], Newton) = {peak_cos:+.6f}")
    print()
    if cos_5 > cos_sgd:
        print(f"     Muon (k=5) is CLOSER to Newton than SGD by {cos_5 - cos_sgd:+.6f}")
    else:
        print(f"     SGD is closer to Newton than Muon (k=5)")
    if cos_5 > cos_adam:
        print(f"     Muon (k=5) is CLOSER to Newton than Adam by {cos_5 - cos_adam:+.6f}")
    else:
        print(f"     Adam is closer to Newton than Muon (k=5)")
    print()

    # ==========================================================================
    # INTERPRETATION
    # ==========================================================================
    print_separator('#')
    print("  INTERPRETATION")
    print_separator('#')
    print()
    if peak_k <= 7 and cos_5 > cos_20:
        print("  The data supports the hypothesis: Muon's approximate orthogonalization (k~5)")
        print("  is NOT a compromise -- it is BETTER than exact orthogonalization for aligning")
        print("  with the Newton direction. This happens because:")
        print("  - At k=0: direction is raw gradient (captures curvature but is poorly conditioned)")
        print("  - At k~5: direction balances gauge-fixing with curvature preservation")
        print("  - At k=20: direction converges to exact orthogonal polar factor,")
        print("    which aggressively projects away curvature information")
        print()
        print("  This is consistent with the RG interpretation: the NS flow has an 'optimal")
        print("  stopping point' where it has removed gauge redundancy without over-projecting")
        print("  curvature-relevant gradient components.")
    elif peak_k <= 7:
        print("  Partial support: The peak IS in the low-k regime, suggesting approximate ortho")
        print("  captures something that exact ortho misses. However, the decline from k=5 to k=20")
        print("  is not significant.")
    else:
        print("  The hypothesis is NOT supported in this setting. Exact orthogonalization")
        print("  does not destroy Newton alignment. The benefit of NS iterations may be")
        print("  purely computational (convergence to orthogonal factor) rather than an")
        print("  information-theoretic sweet spot.")
    print()
    print_separator('#')
    print()

---

## Execute the Experiment

Running the cell below will:
- Train 5 independent deep linear networks for 1500 SGD steps each.
- At 10 checkpoints per seed, compute the full $32 \times 32$ Hessian and Newton direction.
- Evaluate $\cos(\text{ortho}_k(G), d_{\text{Newton}})$ for 10 values of $k$ plus SGD and Adam baselines.
- Print detailed per-seed diagnostics, then aggregate into two summary tables and four hypothesis tests.

**Expected runtime:** Under 1 minute (the Hessian computation is the bottleneck at 64 gradient evaluations per measurement, giving $5 \times 10 \times 64 = 3200$ total gradient evaluations).

In [ ]:
if __name__ == '__main__':
    main()

---

# Part X: Conclusions and Theoretical Implications

## Summary of Results

This experiment measured the directional alignment between Muon's update direction (at varying Newton-Schulz iteration counts $k$) and the exact Newton direction $-H^{\dagger}g$ across a sweep of training checkpoints and random seeds.

### What to look for in the output above:

1. **Inverted-U shape in Table 1:** If $\cos(\text{ortho}_k, \text{Newton})$ rises from $k=0$, peaks around $k=3$--$5$, and then declines for $k \geq 7$, this directly confirms the hypothesis that intermediate orthogonalization is optimal.

2. **$k=5$ vs. $k=20$ gap:** The most critical comparison. If $\cos(k=5) > \cos(k=20)$, then approximate ortho literally *outperforms* exact ortho at tracking second-order curvature. This would mean Muon's practical choice of $k=5$ is not a computational compromise but an information-theoretic optimum.

3. **Muon vs. baselines:** If Muon ($k=5$) beats both SGD and Adam in Newton alignment, it demonstrates that matrix-aware orthogonalization captures cross-parameter curvature structure that neither raw gradients nor diagonal preconditioning can access.

4. **Consistency across training steps:** If the inverted-U shape persists across early (high-loss) and late (low-loss) training checkpoints, the phenomenon is robust rather than regime-specific.

## Theoretical Interpretation: The RG Gauge-Fixing Sweet Spot

The results connect to the Renormalization Group interpretation of Muon as follows:

- **$k=0$ (UV limit):** The raw gradient contains all information but is contaminated by gauge redundancy (components along the Hessian null space from layer-rescaling symmetry). These gauge components are noise from the perspective of optimization.

- **$k \approx 5$ (optimal RG scale):** The NS flow has integrated out the gauge degrees of freedom (the singular value ratios of $G$ carry gauge information) while retaining the singular vector structure that encodes curvature-relevant gradient directions. This is the "effective field theory" that is maximally predictive.

- **$k \to \infty$ (IR limit):** Full orthogonalization forces all singular values to exactly 1, destroying information about the *relative importance* of different gradient directions. The singular values of $G$ partially encode the Hessian eigenstructure (large gradient singular values often correspond to high-curvature directions), and this information is lost.

The inverted-U shape, if confirmed, is the geometric signature of this information-theoretic tradeoff: **too little orthogonalization leaves gauge noise; too much orthogonalization destroys curvature signal.**

## Caveats and Limitations

- **Small scale:** The $4 \times 4$, 2-layer architecture is far from practical network sizes. The gauge structure (16 null directions out of 32 parameters) is proportionally much larger than in real networks.
- **Linear network:** Nonlinear activations introduce additional curvature structure that may shift the optimal $k$.
- **Static measurement:** We measure alignment at fixed checkpoints rather than tracking the dynamic interaction between the optimizer direction and training trajectory.
- **Pseudoinverse sensitivity:** The Newton direction via $H^{\dagger}$ can be sensitive to the `rcond` threshold, especially when the Hessian has eigenvalues near the gauge/non-gauge boundary.

## Connection to Other Experiments

- **Experiment 1.3b (Hessian Null Space):** Verifies that the Hessian indeed has gauge null directions, providing the geometric substrate for this experiment's predictions.
- **Experiment H3 (Normalized SGD vs. Muon):** Shows that orthogonalization's benefit goes beyond mere normalization, consistent with the curvature-preservation mechanism tested here.
- **Experiment H15 (Direction Quality Metric):** Provides an alternative metric for evaluating optimizer directions that complements the Newton-alignment approach used here.